<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Proyect: Este proyecto trata sobre la creación de un sitio web para el juego del Pong

##### Tiempo estimado necesario:  2 meses

El software ofrecerá una interfaz de usuario agradable y capacidades multijugador en tiempo real!

####  Requisitos Técnicos

- 
- 
- 
- 



## Paso 1: Configurar el entorno

Primero creamos una base sólida en contenedores para desplegar todos los servicios de nuestro proyecto. El orquestador que vamos a usar es `docker-compose.yml`

#### Emular Kubernetes (orquestador)

La idea original era usar el orquestador kunbernets por su arquitectura, flexibilidad y escalabiliad. Pero como no lo tenemos instalado en nuestra terminal, vamos a emular una arquitectura kubernets usando `docker-compose.yml`.

### ¿Cómo emulamos Kubernetes con docker-compose?

- Emulamos varios servicios y escalado (por ejemplo, varios contenedores de frontend y backend) usando docker-compose para definir múltiples instancias de los contenedores.
- Usaremos nginx o Traefik para gestionar el balanceo de carga.
Esto nos permitirá simular un entorno cercano a Kubernetes, aunque con limitaciones en temas como escalado automático.




In [ ]:
version: '3.8'

networks:
  transcendence:
    name: transcendence
    driver: bridge

services:
  sqlite:   # Base de datos (SQLite)
    build: dockers/sqlite/.
    container_name: sqlite
    image: nouchka/sqlite3  # Imagen de SQLite para acceder a la base de datos (opcional)
    volumes:
      - sqlite_data:/var/lib/sqlite
    command: tail -f /dev/null  # Mantener el contenedor en ejecución
    networks:
      - transcendence

  backend:    # Backend (Node.js API)
    build: dockers/backend/.
    container_name: app
    image: node:18-alpine
    working_dir: /usr/src/app
    volumes:
      - backend_data:/usr/src/app
      - dockers/backend/data:/usr/src/app/data/.  # Montar la carpeta de datos
    # - ./src:/usr/src/app
    command: npm start
    ports:
      - "3000:3000"
    #depends_on:
    #  - sqlite
    networks:
      - transcendence

  php:    # Servidor PHP (para componentes que aún lo requieran)
    build: dockers/php/.
    container_name: php
    image: php:8.1-fpm
    volumes:
      - php_data:/var/www/html
    networks:
      - transcendence

  frontend:   # Frontend (TypeScript + Tailwind)
    build: dockers/frontend/.
    container_name: frontend
    image: node:18-alpine
    volumes:
      - frontend_data:/usr/src/app
    command: npm run dev
    ports:
      - "8080:8080" # Puerto donde se servirá el frontend
    networks:
      - transcendence

  avalanche:      # Blockchain (Avalanche)
    build: dockers/blockchain/.
    container_name: blockchain
    image: avaplatform/avalanchego
    volumes:
      - blockchain_data:/root/.avalanchego
    ports:
      - "9650:9650"   # JSON RPC endpoint
      - "9651:9651"   # P2P port
    networks:
      - transcendence

  zap:    # Cyberseguridad (OWASP ZAP o similar)
    #build: dockers/security/.
    #container_name: security
    image: owasp/zap2docker-stable
    ports:
      - "8081:8081"
    networks:
      - transcendence   # Puerto para interfaz web de ZAP

volumes:
  sqlite_data:
    name: sqlite_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/sqlite"
      o: bind
  
  backend_data:
    name: backend_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/app"
      o: bind

  php_data:
    name: php_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/php"
      o: bind
  
  frontend_data:
    name: frontend_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/frontend"
      o: bind

  blockchain_data:
    name: blockchain_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/blockchain"
      o: bind

  zap_data:
    name: zap_data
    driver: local
    driver_opts:
      type: none
      device: "users/usuario/data/security"
      o: bind

**Explicación:**
- `Base de datos (SQLite)`: Aunque SQLite no necesita un contenedor en sí, podemos usarlo para gestionar y persistir los datos, aunque también podría ser directamente gestionado por el backend.

- `Backend (Node.js)`: Un contenedor para Node.js que servirá a la API. Se conecta a SQLite usando el volumen compartido y expone el puerto 3000 para la API.

- `Servidor PHP`: Si necesitamos módulos PHP, aquí tenemos un contenedor para ejecutarlos.

- `Frontend (TypeScript + Tailwind)`: Un contenedor que servirá al frontend, donde podemos usar `npm run dev` para servirlo en desarrollo o cambiar a `npm run build` para producción.

- `Blockchain (Avalanche)`: El contenedor simula un nodo Avalanche, donde podemos interactuar con la blockchain y desplegar contratos en Solidity.

- `Cyberseguridad (OWASP ZAP)`: Se incluye OWASP ZAP para monitorear la seguridad de la aplicación en desarrollo.


# Paso 2: Configurar los contenedores

Creamos los contenedores y los almacenamos en los directorios: dockers.

## CONFIGURAR BACKEND:

**Estructura:**
```bash
dockers/backend/
│
├── data/               # Aquí se almacenará el archivo .db de SQLite
├── tools/
│   └── init.sql        # Script para inicializar la base de datos
├── app.js              # Aplicación Node.js
├── Dockerfile          # Dockerfile para la app Node.js
└── package.json        # Configuración del proyecto Node.js
```

### Guia paso a paso instalar la app Node.js.  ###

- 1. Instalar Node.js
Lo primero que necesitamos es instalar Node.js en la máquina. Verificar si ya lo tenemos instalado ejecutando el siguiente comando en tu terminal:
```bash
node -v
```
- 2. Si tienes Node.js instalado, deberías ver un número de versión. Si no, sigue estos pasos: Linux (Ubuntu/Debian):
```bash
sudo apt update
sudo apt install nodejs
sudo apt install npm
```
- 3. Crear proyecto Node.js<br/>
Crea un directorio:
```bash
mkdir backend
cd backend
```
Inicializa un proyecto de Node.js - Esto creará `package.json`, que es donde se definen las dependencias y la configuración del proyecto.:
```bash
npm init -y
```
Instalar las dependencias necesarias:

Necesitamos algunas dependencias básicas para la app, como el framework Express para crear la API. Ejecuta los siguientes comandos para instalar:

· Express: Un framework para manejar rutas y peticiones HTTP.<br>
· sqlite3: Para conectar y trabajar con SQLite.
```bash
npm install express sqlite3
```
Crear el archivo principal de la app:

Creamos un archivo que será el punto de entrada de la aplicación. Lo llamaremos app.js.
```bash
touch app.js
```
Escribir la lógica básica del servidor:

Abre app.js con tu editor de texto y escribe el siguiente código para configurar el servidor Node.js con Express y conectarlo a SQLite:







In [ ]:
const express = require('express');
const sqlite3 = require('sqlite3').verbose();
const fs = require('fs');
const path = require('path');

const app = express();
const port = 3000;

// Conectar a la base de datos SQLite
const dbPath = path.join(__dirname, 'data', 'sqlite.db');
const db = new sqlite3.Database(dbPath);

// Ejecutar el script de inicialización de la base de datos
const initSQL = fs.readFileSync(path.join(__dirname, 'tools', 'init.sql'), 'utf-8');
db.exec(initSQL, (err) => {
    if (err) {
        console.error('Error al inicializar la base de datos:', err.message);
    } else {
        console.log('Base de datos inicializada correctamente');
    }
});

// Ruta básica para probar el servidor
app.get('/', (req, res) => {
    res.send('¡Hola, mundo desde Node.js!');
});

// Iniciar el servidor
app.listen(port, () => {
    console.log(`Servidor escuchando en http://localhost:${port}`);
});


En este archivo, la aplicación de Node.js:

Se conecta a una base de datos SQLite.<br>
Ejecuta el script de inicialización `init.sql` desde la carpeta tools.<br>
Escucha en el puerto `3000` y responde a la ruta /.

Crear la estructura de directorios:

Crea la estructura de carpetas que necesitas para el proyecto, como tools y data:
```bash
mkdir tools
mkdir data
```
Luego, coloca tu archivo init.sql en el directorio tools.

Probar la app localmente:

Ahora, podemos ejecutar la aplicación para asegurarnos de que funciona correctamente.

In [ ]:
node app.js

Si todo está bien, deberías ver el mensaje en la consola:

In [ ]:
Servidor escuchando en http://localhost:3000

#### NOTA: IMPORTANTE ####

No es necesario tener todos los node_modules dentro de tu contenedor para cada vez que realicemos una compilación o ejecución. Una buena práctica para reducir el tamaño y mejorar el rendimiento de la aplicación es:

Ignorar node_modules en el volumen local: 
- En lugar de montar node_modules como un volumen en la máquina local (lo cual puede ocupar mucho espacio), podemos instalar las dependencias directamente dentro del contenedor y no compartir el directorio con la máquina host.

- Usar `.dockerignore`: Creamos un archivo `.dockerignore` para evitar que los node_modules del host se copien dentro del contenedor al construir la imagen. Esto mantendrá limpio el contenedor.

- Instalación de dependencias solo en el contenedor: En lugar de instalar las dependencias en tu máquina local, puedes hacerlo en el contenedor durante la construcción de la imagen.

Paso 1: Crear el archivo .dockerignore
Dentro del directorio donde tengas el Dockerfile para tu backend (dockers/app/), crea un archivo llamado .dockerignore y añade lo siguiente:
```bash
node_modules
npm-debug.log
.env
```
Esto evitará que node_modules y otros archivos innecesarios sean copiados al contenedor durante la construcción.

Paso 2: Modificar el Dockerfile para instalar dependencias dentro del contenedor

In [ ]:
# Dockerfile for Node.js app
FROM node:18-alpine

# Set working directory
WORKDIR /usr/src/app

# Copy package.json and package-lock.json to install dependencies
COPY package*.json ./

# Install dependencies
RUN npm install --production

# Copy the rest of the application code
COPY . .

# Expose the port on which the app runs
EXPOSE 3000

# Start the application
CMD ["npm", "start"]


Paso 3: Volúmenes en docker-compose.yml

Si decidimoss no compartir node_modules con el host, no necesitamos montarlo como volumen. Asegúraerse de que el servicio del backend en docker-compose.yml tenga algo como esto:

In [ ]:
backend:    # Backend (Node.js API)
    build: dockers/backend/.
    container_name: app
    image: node:18-alpine
    working_dir: /usr/src/app
    volumes:
      - backend_data:/usr/src/app
      - dockers/backend/data:/usr/src/app/data/.  # Montar la carpeta de datos
    # - ./src:/usr/src/app
    command: npm start
    ports:
      - "3000:3000"
    #depends_on:
    #  - sqlite
    networks:
      - transcendence

## Situación actual:

No tienes ningún directorio `node_modules` en dockers/backend en tu máquina local. Eso está bien.

Cuando levantes el contenedor, efectivamente, se instalarán las dependencias. Pero, ¿dónde?

¿Dónde se instalan los node_modules?

Cuando ejecutas tu contenedor con Docker, el proceso de instalación de dependencias (npm install) ocurre dentro del contenedor. Específicamente, los archivos node_modules se crearán en el sistema de archivos del contenedor. No se crearán ni existirán en tu máquina local a menos que los montes de manera explícita con un volumen.

Flujo de instalación en Docker
Dockerfile:

Cuando Docker construye la imagen desde tu Dockerfile, copia el archivo `package.json` y luego ejecuta `RUN npm install --production`. Esto instalará las dependencias de producción dentro del contenedor, en el directorio de trabajo /usr/src/app (que es el WORKDIR especificado en el Dockerfile).
Dentro del contenedor:

Después de ejecutar `npm install`, los archivos node_modules estarán disponibles en `/usr/src/app/node_modules` dentro del contenedor, pero no en tu máquina local (a menos que hayas configurado volúmenes que monten esa carpeta de manera explícita, lo cual no parece ser el caso).


In [ ]:
volumes:
    - backend_data:/usr/src/app
    - dockers/backend/data:/usr/src/app/data/

1. `backend_data:/usr/src/app`: Este volumen monta el directorio backend_data de tu sistema local en /usr/src/app dentro del contenedor. Lo que significa que lo que ocurra dentro del contenedor en `/usr/src/app` se refleja en el volumen backend_data.

- Esto incluiría los node_modules, ya que `/usr/src/app` es donde estás ejecutando npm install. Así que los node_modules también se montarán en este volumen.

**Solución** si no quieres que node_modules se guarden en el volumen: Puedes montar solo partes específicas de tu aplicación, como tu código fuente o la carpeta de datos, pero no el directorio node_modules. Para hacer esto, ajusta el volumen para que excluya node_modules.

In [ ]:
volumes:
    - backend_data:/usr/src/app
    - dockers/backend/data:/usr/src/app/data/
    - /usr/src/app/node_modules  # Excluyendo node_modules

Esto garantiza que node_modules se mantenga dentro del contenedor y no se monte en tu sistema local.

2. `dockers/backend/data:/usr/src/app/data/`: Este volumen solo monta la carpeta de datos de SQLite

<br>

## CONFIGURAR SQLite:

1. Estructura para SQLite

SQLite almacena su base de datos en un archivo, así que no necesitamos un servidor separado. La base de datos simplemente se almacena en un archivo .db.

2. Archivo SQL para inicializar la base de datos:

En la carpeta tools, creamos el archivo init.sql, que contiene las instrucciones para crear las tablas necesarias y cualquier dato de inicio que se necesite.

In [ ]:
-- init.sql en tools/
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    password TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS games (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    player1_id INTEGER,
    player2_id INTEGER,
    winner_id INTEGER,
    FOREIGN KEY (player1_id) REFERENCES users (id),
    FOREIGN KEY (player2_id) REFERENCES users (id),
    FOREIGN KEY (winner_id) REFERENCES users (id)
);


3. Incluir SQLite en la app Node.js:

Ya hamos instalado sqlite3 en la aplicación Node.js, así que ahora solo necesitamos asegurarnos de que todo esté conectado.

Revisar archivo app.js para asegurar de que está conectando correctamente con SQLite y ejecutando el script init.sql cuando la app arranca.


4. Dockerfile para la app con SQLite

No necesitamos un servicio separado para SQLite, ya que se gestiona como parte de la aplicación Node.js.

El Dockerfile para Node.js así debería estar bien

In [ ]:
# Usar una imagen base de Node.js
FROM node:18-alpine

# Crear y establecer el directorio de trabajo
WORKDIR /usr/src/app

# Copiar el package.json y package-lock.json
COPY package*.json ./

# Instalar dependencias
RUN npm install

# Copiar el resto de los archivos de la app
COPY . .

# Crear la carpeta para la base de datos SQLite dentro del contenedor
RUN mkdir -p /usr/src/app/data

# Exponer el puerto de la app
EXPOSE 3000

# Comando para ejecutar la app
CMD ["node", "app.js"]


Este `Dockerfile` también asegura que se cree una carpeta data dentro del contenedor para almacenar la base de datos SQLite.

5. Inicializar la base de datos en el contenedor

La aplicación Node.js ejecuta automáticamente el script init.sql la primera vez que se conecta a la base de datos, por lo que cuando levantes los contenedores, la base de datos se inicializará correctamente dentro del contenedor.

<br>

## CONFIGURAR: frontend funcione con TypeScript y Tailwind CSS

1. Estructura del Proyecto

El frontend debe estar dentro del directorio dockers/frontend/, que es el que hemos mencionado en el docker-compose.yml. Si no lo tienes creado, hazlo ahora.

Estructura esperada del proyecto:

In [ ]:
/dockers
  /frontend
    /src
    /public
    Dockerfile
    package.json
    tsconfig.json
    tailwind.config.js


**Explicación:**
- `src/`: Aquí van los archivos de código TypeScript (por ejemplo, index.ts).
- `public/`: Aquí van los archivos estáticos, como el index.html. <font color="green">ERROR CORREGIDO</font>
- `Dockerfile`: El archivo que configura el contenedor para tu frontend.
- `package.json`: Donde se definen las dependencias de Node.js, incluidas TypeScript y Tailwind.
- `tsconfig.json`: Archivo de configuración para TypeScript.
- `tailwind.config.js`: Archivo de configuración para Tailwind.

2. Crear el Dockerfile para el Frontend
Dentro de dockers/frontend/, crea un archivo Dockerfile con el siguiente contenido:

In [ ]:
# Usar una imagen base de Node.js
FROM node:18-alpine

# Establecer el directorio de trabajo en el contenedor
WORKDIR /usr/src/app

# Copiar package.json y package-lock.json
COPY package*.json ./

# Instalar las dependencias de npm (incluye TypeScript y Tailwind)
RUN npm install

# Copiar todo el código del frontend al contenedor
COPY . .

# Exponer el puerto donde se servirá el frontend
EXPOSE 3001

# Comando para iniciar el servidor de desarrollo
CMD ["npm", "run", "dev"]


3. Crear el package.json
Dentro del mismo directorio dockers/frontend/, crea un archivo `package.json` con el siguiente contenido:

In [ ]:
{
  "name": "frontend",
  "version": "1.0.0",
  "scripts": {
    "dev": "vite",
    "build": "vite build"
  },
  "dependencies": {
    "tailwindcss": "^3.0.0",
    "vite": "^2.0.0",
    "typescript": "^4.4.0"
  },
  "devDependencies": {
    "autoprefixer": "^10.0.0",
    "postcss": "^8.0.0"
  }
}


5. Configuración de TypeScript
También necesitas configurar TypeScript. Crea un archivo `tsconfig.json` dentro de dockers/frontend/ con el siguiente contenido:

In [ ]:
{
    "compilerOptions": {
      "target": "esnext",
      "module": "esnext",
      "moduleResolution": "node",
      "strict": true,
      "jsx": "react-jsx",
      "esModuleInterop": true,
      "skipLibCheck": true,
      "forceConsistentCasingInFileNames": true
    },
    "include": [
      "src/**/*"
    ]
  }
  

Este archivo configura TypeScript para trabajar con la última versión de JavaScript y React (si decides usarlo en el futuro).

6. Crear Archivos de Frontend
Ahora, en el directorio dockers/frontend/src/, crea un archivo `index.ts` (podemos agregar más archivos según lo necesitesmos):

In [ ]:
console.log('Hola desde TypeScript!');

Y en dockers/frontend/public/, crea un archivo index.html con este contenido básico:

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Frontend con TypeScript y Tailwind CSS</title>
  <link href="/dist/output.css" rel="stylesheet">
</head>
<body class="bg-gray-100">
  <h1 class="text-4xl font-bold text-center mt-10">¡Hola, mundo!</h1>
  <script type="module" src="/src/index.ts"></script>
</body>
</html>

7. Acceder al Frontend

Accede al frontend desde tu navegador en http://localhost:8080.
¡Y listo! Ahora tenemos el frontend con TypeScript y Tailwind CSS funcionando dentro de Docker.

Resumen de Pasos:
- Crear Dockerfile en dockers/frontend/ para configurar el contenedor de frontend.
- Crear package.json con las dependencias necesarias (TypeScript, Tailwind, etc.).
- Configurar tailwind.config.js y tsconfig.json para habilitar Tailwind y TypeScript.
- Crear archivos frontend (como index.ts y index.html).
- Levantar los contenedores con docker-compose up --build.
- Acceder al frontend en http://localhost:8080.


<br>

## CONFIGURAR: Security (OWASP ZAP)

Pasos a seguir:

1. **Crear carpeta** para las reglas de escaneo: En tu proyecto, asegúrate de tener una carpeta tools/zap_rules donde coloques los scripts de reglas o configuraciones que ZAP debe usar para automatizar los escaneos.

2. **`docker-compose.yml`** Asegúrate de incluir la configuración de OWASP ZAP en tu `docker-compose.yml` para que se ejecute dentro de tu red de contenedores:


In [ ]:
security:
build: dockers/security/.  # Ruta al Dockerfile de OWASP ZAP
container_name: security
volumes:
  - ./tools/zap_rules:/zap/wrk/  # Montar la carpeta con las reglas de escaneo
ports:
  - "8081:8081"  # Puerto para acceder a OWASP ZAP
networks:
  - transcendence

3. Crear reglas de escaneo para OWASP ZAP

- Crear un script básico de escaneo
OWASP ZAP permite utilizar su API para automatizar los escaneos. Puedes crear un script que use la API para definir cómo quieres que se realicen los escaneos.

zap_scan.sh:

In [ ]:
#!/bin/bash

# URL del objetivo que deseas escanear
TARGET_URL="http://tuapp.local"

# Dirección y puerto de OWASP ZAP (esto depende de tu configuración)
ZAP_ADDRESS="http://localhost"
ZAP_PORT="8081"

# Iniciar un escaneo pasivo
echo "Iniciando escaneo pasivo..."
curl "$ZAP_ADDRESS:$ZAP_PORT/JSON/ascan/action/scan/?url=$TARGET_URL"

# Puedes esperar a que termine el escaneo si quieres
# sleep 10  # Esperar 10 segundos para que el escaneo progrese

# Descargamos los resultados del escaneo
echo "Descargando resultados..."
curl "$ZAP_ADDRESS:$ZAP_PORT/OTHER/core/other/htmlreport/" -o zap_report.html

echo "Escaneo completado. El reporte está guardado en zap_report.html"


Este script usa cURL para interactuar con la API de OWASP ZAP en su modo daemon y realizar un escaneo. Reemplaza TARGET_URL con la URL de tu aplicación.

- Guardar el script en la carpeta `tools/zap_rules`
- Crea un directorio en tu proyecto llamado `tools/zap_rules`.
- Guarda el script anterior en esa carpeta, por ejemplo, como `zap_scan.sh`.
- Asegúrate de que el archivo tenga permisos de ejecución.

In [ ]:
chmod +x tools/zap_rules/zap_scan.sh

4. Configurar el Dockerfile para usar el script

Tu Dockerfile debe copiar los scripts que acabas de crear dentro del contenedor OWASP ZAP.

In [ ]:
# Dockerfile para OWASP ZAP
FROM owasp/zap2docker-stable

# Copiar el script de reglas de escaneo desde la carpeta tools a /zap/wrk/
COPY tools/zap_rules/zap_scan.sh /zap/wrk/zap_scan.sh

# Dar permisos de ejecución al script dentro del contenedor
RUN chmod +x /zap/wrk/zap_scan.sh

# Exponer el puerto 8081 para la interfaz web de ZAP
EXPOSE 8081

# Iniciar OWASP ZAP en modo daemon, accesible desde todas las interfaces
# Esto ejecutará OWASP ZAP en modo daemon y automáticamente ejecutará el script de escaneo.
CMD ["zap.sh", "-daemon", "-host", "0.0.0.0", "-port", "8081", "&&", "/zap/wrk/zap_scan.sh"]

## Authors


- David Gallego "davgalle"
- Nicolas Gonzalez de Mendoza "nicgonza"


Copyright © ErMichoss Corporation. All rights reserved.
